# Phase 1: Data Foundation — Step 1.4: Data Relationships

This notebook joins `occupation_master.csv`, `essential_skills_processed.csv`, and `software_skills_processed.csv` on `O*NET-SOC Code` to create a unified `role_skills.csv` mapping occupations to both essential and software skills in a single long-format table.

In [1]:
import os
import pandas as pd

proc_dir = os.path.join("data", "processed")
print(f"Reading processed data from: {os.path.abspath(proc_dir)}")

Reading processed data from: C:\Users\Harshit Mishra\OneDrive\Desktop\enterprise_hr_ai\data\processed


## 1. Load Processed Taxonomy and Skills Datasets

In [2]:
df_occ = pd.read_csv(os.path.join(proc_dir, "occupation_master.csv"))
df_ess = pd.read_csv(os.path.join(proc_dir, "essential_skills_processed.csv"))
df_soft = pd.read_csv(os.path.join(proc_dir, "software_skills_processed.csv"))

print(f"Occupation Master Shape: {df_occ.shape}")
print(f"Essential Skills Shape: {df_ess.shape}")
print(f"Software Skills Shape: {df_soft.shape}")

Occupation Master Shape: (1016, 3)
Essential Skills Shape: (18200, 15)
Software Skills Shape: (31821, 7)


## 2. Process and Align Essential Skills
We filter essential skills to only include the `Importance` scale. We then map `Element Name` to `Skill Name` and use `Data Value` as the `Relevance Score`.

In [3]:
df_ess_imp = df_ess[df_ess["Scale Name"] == "Importance"].copy()
df_ess_imp_aligned = df_ess_imp[["O*NET-SOC Code", "Element Name", "Data Value"]].copy()
df_ess_imp_aligned.rename(columns={"Element Name": "Skill Name", "Data Value": "Relevance Score"}, inplace=True)
df_ess_imp_aligned["Skill Type"] = "Essential"
print(f"Aligned Essential Skills Shape: {df_ess_imp_aligned.shape}")

Aligned Essential Skills Shape: (9100, 4)


## 3. Process and Align Software Skills
We map `Workplace Example` to `Skill Name`. We calculate the `Relevance Score` on a 3.0 to 5.0 scale using: `3.0 + 1.0 (if Hot Technology == 'Y') + 1.0 (if In Demand == 'Y')`.

In [4]:
df_soft_aligned = df_soft[["O*NET-SOC Code", "Workplace Example"]].copy()
df_soft_aligned.rename(columns={"Workplace Example": "Skill Name"}, inplace=True)
df_soft_aligned["Skill Type"] = "Software"

# Relevance logic based on demand/technology flags
df_soft_aligned["Relevance Score"] = (
    3.0 
    + (df_soft["Hot Technology"] == "Y").astype(float) 
    + (df_soft["In Demand"] == "Y").astype(float)
)
print(f"Aligned Software Skills Shape: {df_soft_aligned.shape}")

Aligned Software Skills Shape: (31821, 4)


## 4. Concatenate and Join with Occupation Master
We concatenate the essential and software skills, and then merge with `occupation_master` on `O*NET-SOC Code` using an inner join to obtain job titles and descriptions.

In [5]:
df_skills_combined = pd.concat([df_ess_imp_aligned, df_soft_aligned], ignore_index=True)
df_role_skills = pd.merge(df_occ, df_skills_combined, on="O*NET-SOC Code", how="inner")
print(f"Final Joined role_skills Shape: {df_role_skills.shape}")
print(f"Unique O*NET Codes in role_skills: {df_role_skills['O*NET-SOC Code'].nunique()}")

Final Joined role_skills Shape: (40921, 6)
Unique O*NET Codes in role_skills: 923


## 5. Save the Combined Dataset
We save the consolidated role skills mapping to `data/processed/role_skills.csv`.

In [6]:
output_path = os.path.join(proc_dir, "role_skills.csv")
df_role_skills.to_csv(output_path, index=False)
print(f"Successfully saved consolidated role_skills to {output_path}")

Successfully saved consolidated role_skills to data\processed\role_skills.csv
